In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import phoenix as px_app
from llama_index.core import Settings
from llama_index.llms.groq import Groq
from dotenv import load_dotenv
from openinference.instrumentation.llama_index import LlamaIndexInstrumentor

load_dotenv('../.env') 

# Launch Phoenix Dashboard
session = px_app.launch_app()
print(f"🌍 Phoenix Dashboard is running at: {session.url}")

# Set up OpenTelemetry with Phoenix
from opentelemetry import trace as trace_api
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk import trace as trace_sdk
from opentelemetry.sdk.trace.export import SimpleSpanProcessor

# Configure tracer to send to Phoenix
tracer_provider = trace_sdk.TracerProvider()
tracer_provider.add_span_processor(SimpleSpanProcessor(OTLPSpanExporter(endpoint="http://localhost:6006/v1/traces")))
trace_api.set_tracer_provider(tracer_provider)

# Instrument LlamaIndex BEFORE any LLM calls
LlamaIndexInstrumentor().instrument()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("❌ API Key missing! Check your .env file.")

llm = Groq(model="llama-3.3-70b-versatile", api_key=api_key)
print(f"✅ Setup Complete. Dashboard running at: {session.url}")

try:
    df = pd.read_csv('../data/RA_Application_Task.csv')
    print(f"✅ Loaded {len(df)} housing records.")
except FileNotFoundError:
    print("❌ Error: Could not find the CSV file. Please check the 'data' folder.")

def get_ai_estimate(row):
    """
    Sends home details to Llama 3 and extracts a price.
    """
    prompt = (
        f"Act as a real estate expert. Estimate the fair market value of a home with these details:\n"
        f"- Bedrooms: {row['Bedrooms']}\n"
        f"- Bathrooms: {row['Bathrooms Comparable']}\n"
        f"- Lot Size: {row['LotSize Comparable']} sqft\n"
        f"- Year Built: {row['YearBuilt Comparable']}\n\n"
        f"Provide ONLY the estimated price as a single number (e.g., 250000). "
        f"Do not output any text, currency symbols ($), or explanations."
    )
    
    try:
        response = llm.complete(prompt)
        price_text = response.text.strip().replace('$', '').replace(',', '')
        return float(price_text)
    except Exception as e:
        print(f"⚠️  Error: {e}")
        return np.nan

print("🚀 Starting AI Valuation (This takes about 30-60 seconds)...")

df['AI_Estimated_Value'] = df.apply(get_ai_estimate, axis=1)

df['Estimation_Diff'] = df['AI_Estimated_Value'] - df['Comparable Sale Price']

results = df.dropna(subset=['AI_Estimated_Value'])

fig = go.Figure()
fig.add_trace(go.Scatter(x=results.index, y=results['Comparable Sale Price'],
                         mode='lines+markers', name='Actual Price', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=results.index, y=results['AI_Estimated_Value'],
                         mode='lines+markers', name='AI Estimate', line=dict(color='red', dash='dot')))

fig.update_layout(title="Llama 3 vs. Actual Market Prices", xaxis_title="House ID", yaxis_title="Price ($)")

# Ensure outputs directory exists
os.makedirs('../outputs', exist_ok=True)

fig.write_html("../outputs/performance_curve.html")
results.to_csv("../outputs/experiment_results.csv", index=False)

print("\n🎉 SUCCESS!")
print(f"1. Observability Dashboard: {session.url} (Click this for Screenshot!)")
print(f"2. Graph Saved: outputs/performance_curve.html")